<a href="https://colab.research.google.com/github/pradhapmoorthi/CVND/blob/Trial/2_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step.
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file.
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:**


### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:**

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters())
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:**

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:**

## 1. Basic Setup

In [48]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/cvnd_coco"
os.makedirs(DRIVE_ROOT, exist_ok=True)

print("✅ DRIVE_ROOT:", DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ DRIVE_ROOT: /content/drive/MyDrive/cvnd_coco


In [49]:
!pip -q install nltk pycocotools tqdm

In [50]:
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [51]:
import os
import sys
import urllib.request
from urllib.error import HTTPError

CODE_DIR = "/content/code"
os.makedirs(CODE_DIR, exist_ok=True)

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/pradhapmoorthi/CVND/Trial"

def download_or_skip(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print(f"✅ exists: {dst}")
        return True
    try:
        print(f"⬇️ downloading: {url}")
        urllib.request.urlretrieve(url, dst)
        print(f"✅ saved: {dst}")
        return True
    except HTTPError as e:
        print(f"⚠️ HTTP error for {url}: {e}")
        return False

# Required files
ok1 = download_or_skip(f"{GITHUB_RAW_BASE}/model.py",       os.path.join(CODE_DIR, "model.py"))
ok2 = download_or_skip(f"{GITHUB_RAW_BASE}/data_loader.py", os.path.join(CODE_DIR, "data_loader.py"))

# Vocabulary file name sometimes differs; try both
ok3 = download_or_skip(f"{GITHUB_RAW_BASE}/vocabulary.py",  os.path.join(CODE_DIR, "vocabulary.py"))
if not ok3:
    ok3b = download_or_skip(f"{GITHUB_RAW_BASE}/vocalbulary.py", os.path.join(CODE_DIR, "vocabulary.py"))
    if not ok3b:
        raise RuntimeError("Could not download vocabulary file (tried vocabulary.py and vocalbulary.py).")

# Add to sys.path so imports work
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print("✅ CODE_DIR ready:", CODE_DIR)
!ls -l /content/code

⬇️ downloading: https://raw.githubusercontent.com/pradhapmoorthi/CVND/Trial/model.py
✅ saved: /content/code/model.py
⬇️ downloading: https://raw.githubusercontent.com/pradhapmoorthi/CVND/Trial/data_loader.py
✅ saved: /content/code/data_loader.py
⬇️ downloading: https://raw.githubusercontent.com/pradhapmoorthi/CVND/Trial/vocabulary.py
✅ saved: /content/code/vocabulary.py
✅ CODE_DIR ready: /content/code
total 28
-rw-r--r-- 1 root root  6982 Apr 20 20:26 data_loader.py
-rw-r--r-- 1 root root 13882 Apr 20 20:26 model.py
-rw-r--r-- 1 root root  3598 Apr 20 20:26 vocabulary.py


In [ ]:
import os
import urllib.request
import zipfile

COCO_ROOT = "/content/cocoapi"
os.makedirs(COCO_ROOT, exist_ok=True)
os.makedirs(f"{COCO_ROOT}/images", exist_ok=True)
os.makedirs(f"{COCO_ROOT}/annotations", exist_ok=True)

TRAIN_IMG_URL = "http://images.cocodataset.org/zips/train2014.zip"
ANN_URL       = "http://images.cocodataset.org/annotations/annotations_trainval2014.zip"

TRAIN_ZIP = "/content/train2014.zip"
ANN_ZIP   = "/content/annotations_trainval2014.zip"

def download_if_missing(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print("✅ zip exists:", dst)
        return
    print("⬇️ downloading:", url)
    urllib.request.urlretrieve(url, dst)
    print("✅ downloaded:", dst)

def unzip_if_needed(zip_path, out_dir, expected_path=None):
    if expected_path and os.path.exists(expected_path):
        print("✅ already extracted:", expected_path)
        return
    print("📦 extracting:", zip_path, "->", out_dir)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)
    print("✅ extracted")

download_if_missing(TRAIN_IMG_URL, TRAIN_ZIP)
download_if_missing(ANN_URL, ANN_ZIP)

# Extract train images into COCO_ROOT/images (creates train2014 folder)
unzip_if_needed(TRAIN_ZIP, f"{COCO_ROOT}/images", expected_path=f"{COCO_ROOT}/images/train2014")

# Extract annotations into COCO_ROOT (creates annotations folder with captions_train2014.json)
unzip_if_needed(ANN_ZIP, COCO_ROOT, expected_path=f"{COCO_ROOT}/annotations/captions_train2014.json")

print("\n✅ COCO structure check:")
!ls -l /content/cocoapi
!ls -l /content/cocoapi/images | head
!ls -l /content/cocoapi/annotations | grep captions | head

In [ ]:
import os

train_img_dir = "/content/cocoapi/images/train2014"
cap_file = "/content/cocoapi/annotations/captions_train2014.json"

assert os.path.isdir(train_img_dir), f"Missing train image dir: {train_img_dir}"
assert os.path.isfile(cap_file), f"Missing captions file: {cap_file}"

print("✅ Paths OK")
print("Train images:", len(os.listdir(train_img_dir)))
print("Captions file:", cap_file)

In [ ]:
import torchvision.transforms as transforms
from data_loader import get_loader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))
])

VOCAB_FILE = f"{DRIVE_ROOT}/vocab.pkl"

data_loader = get_loader(
    transform=transform,
    mode="train",
    batch_size=128,
    vocab_threshold=5,
    vocab_file=VOCAB_FILE,
    vocab_from_file=False,  # ✅ FIRST RUN: build vocab
    num_workers=2,
    cocoapi_loc="/content"
)

print("✅ Dataset size:", len(data_loader.dataset))
print("✅ Vocab size:", len(data_loader.dataset.vocab))
print("✅ Vocab saved/used at:", VOCAB_FILE)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from model import EncoderCNN, DecoderRNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Device:", device)

embed_size = 256
hidden_size = 512
num_layers = 1
vocab_size = len(data_loader.dataset.vocab)

encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)

print("✅ Model initialized")

In [ ]:
import time
import os

num_epochs = 5
log_every = 50

for epoch in range(1, num_epochs + 1):
    encoder.train()
    decoder.train()

    running_loss = 0.0
    t0 = time.time()

    print(f"\n===== Epoch {epoch}/{num_epochs} =====")

    for step, (images, captions) in enumerate(data_loader):
        images = images.to(device)
        captions = captions.to(device)

        # Forward
        features = encoder(images)
        #outputs = decoder(features, captions)
        outputs, alphas = decoder(features, captions)

        # Targets: next-word prediction
        targets = captions[:, 1:].reshape(-1)

        # Make outputs 2D for CrossEntropyLoss
        if outputs.dim() == 3:
            outputs = outputs.reshape(-1, outputs.size(-1))

        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (step + 1) % log_every == 0:
            avg = running_loss / log_every
            print(f"Step {step+1}/{len(data_loader)}  AvgLoss={avg:.4f}  Elapsed={time.time()-t0:.1f}s")
            running_loss = 0.0
            t0 = time.time()

    # Save checkpoint each epoch
    ckpt_path = os.path.join(DRIVE_ROOT, f"ckpt_epoch{epoch}.pt")
    torch.save({
        "epoch": epoch,
        "encoder_state_dict": encoder.state_dict(),
        "decoder_state_dict": decoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "embed_size": embed_size,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "vocab_file": VOCAB_FILE,
    }, ckpt_path)

    print("✅ Saved checkpoint:", ckpt_path)

print("\n🎉 Training complete")

<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here.

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.